In [43]:
from pathlib import Path

import pandas as pd
import polars as pl
import polars_h3 as plh3
import numpy as np
import more_itertools
from datetime import date
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [44]:
RES = 7
VARIANT = "v2"  # of stop detection

In [45]:
path = f"{config['dirs']['output']}/{config['city']}"
Path(f"{path}/hexagon_network/{VARIANT}/{RES}/partials").mkdir(
    exist_ok=True, parents=True
)

In [46]:
dummy_seq = pd.DataFrame(
    {
        "device_id": [1] * 7,
        "day": [date(2026, 5, 19)] * 5 + [date(2026, 5, 20)] * 2,
        "h3_7": [1, 2, 3, 2, 3, 1, 2],
        "latitude_mean": [47.42, 47.64, 47.23, 47.87, 47.25, 47.42, 47.64],
        "longitude_mean": [18.91, 19.12, 19.09, 19.31, 19.08, 18.91, 19.12],
    }
)

In [35]:
# def get_daily_stop_sequence(seq, res: int = 7) -> dict | float:
#     result = {}
#     for i, j in more_itertools.pairwise(seq[f"h3_{res}"]):
#         if i == j or pd.isna(i) or pd.isna(j):
#             continue
#         if i not in result:
#             result[i] = {}
#         if j not in result[i]:
#             result[i][j] = 0
#         result[i][j] += 1

#     if len(result) == 0:
#         return np.nan
#     else:
#         return result


# get_daily_stop_sequence(dummy_seq)

In [47]:
def load_data_pl(i: int, res: int, path: str) -> pl.DataFrame:
    data = pl.read_parquet(f"{path}/{i:02x}.parquet")
    data = data.with_columns(
        plh3.latlng_to_cell(
            pl.col("latitude_mean"), pl.col("longitude_mean"), res, return_dtype=pl.Utf8
        ).alias(f"h3_{res}"),
        pl.col("interval_start").dt.to_string("%Y-%m-%d").alias("day"),
        pl.col("interval_start").dt.to_string("%H").alias("hour"),
    )

    data = data.sort("device_id", "interval_start", descending=[False, False])
    return data


def get_daily_stop_sequence_pl(seq: pl.DataFrame, hexagon_column: str) -> dict:
    transitions = (
        seq.select(
            [
                pl.col(hexagon_column).alias("src"),
                pl.col(hexagon_column).shift(-1).alias("dst"),
            ]
        )
        .filter(
            (pl.col("src") != pl.col("dst"))
            & pl.col("src").is_not_null()
            & pl.col("dst").is_not_null()
        )
        .group_by(["src", "dst"])
        .len()
    )

    result = {}

    for row in transitions.iter_rows(named=True):
        src, dst, count = row["src"], row["dst"], row["len"]

        result.setdefault(src, {})[dst] = count

    return result


get_daily_stop_sequence_pl(pl.from_pandas(dummy_seq), "h3_7")

{2: {3: 2}, 3: {2: 1, 1: 1}, 1: {2: 2}}

In [37]:
# (
#     pl.from_pandas(dummy_seq)
#     .with_columns([pl.col("h3_7").shift(-1).over(["device_id", "day"]).alias("target")])
#     .filter((pl.col("h3_7") != pl.col("target")) & pl.col("target").is_not_null())
#     .group_by(["device_id", "day", "h3_7", "target"])
#     .len()
# ).rename({"h3_7": "source", "len": "weight"}).sort(["device_id", "day"])

In [70]:
col = f"h3_{RES}"
for i in range(256):
    stops = load_data_pl(i, RES, f"{path}/stops/{VARIANT}")
    connections = (
        (
            stops.with_columns(
                [pl.col(col).shift(-1).over(["device_id", "day"]).alias("target")]
            )
            .filter((pl.col(col) != pl.col("target")) & pl.col("target").is_not_null())
            .group_by(["device_id", "day", "hour", col, "target"])
            .len()
        )
        .rename({col: "source", "len": "weight"})
        .sort(["device_id", "day", "hour"])
    )
    connections.write_parquet(
        f"{path}/hexagon_network/{VARIANT}/{RES}/partials/connections_{i:02x}.parquet"
    )

# 1m 43s for level 7
# 1m 37s for level 8
# 1m 38s for level 9
# 1m 37s for level 10

In [39]:
connections = pl.DataFrame()
for i in range(256):
    temp = pl.read_parquet(
        f"{path}/hexagon_network/{VARIANT}/{RES}/partials/connections_{i:02x}.parquet"
    )
    connections = pl.concat([connections, temp])
    del temp

In [40]:
network = (
    connections.group_by(["source", "target"])
    .agg(pl.col("weight").sum())
    .sort("source", "target")
)
network.write_csv(f"{path}/hexagon_network/{VARIANT}/{RES}/edges_trip.csv")

In [41]:
network_by_unique_device = (
    connections.group_by(["source", "target"])
    .agg(pl.col("device_id").unique().len().alias("weight"))
    .sort("source", "target")
)
network_by_unique_device.write_csv(
    f"{path}/hexagon_network/{VARIANT}/{RES}/edges_device.csv"
)

add hour

In [71]:
col = f"h3_{RES}"
conns_with_hour = pl.DataFrame()
for i in range(256):
    stops = load_data_pl(i, RES, f"{path}/stops/{VARIANT}")
    connections = (
        (
            stops.with_columns(
                [pl.col(col).shift(-1).over(["device_id", "day"]).alias("target")]
            )
            .filter((pl.col(col) != pl.col("target")) & pl.col("target").is_not_null())
            .group_by(["device_id", "day", "hour", col, "target"])
            .len()
        )
        .rename({col: "source", "len": "weight"})
        .sort(["device_id", "day", "hour"])
    )
    # connections_ = (
    #     (
    #         stops.with_columns(
    #             [pl.col(col).shift(-1).over(["device_id", "day"]).alias("target")]
    #         )
    #         .filter((pl.col(col) != pl.col("target")) & pl.col("target").is_not_null())
    #         .group_by(["device_id", "day", col, "target"])
    #         .len()
    #     )
    #     .rename({col: "source", "len": "weight"})
    #     .sort(["device_id", "day"])
    # )
    conns_with_hour = pl.concat([conns_with_hour, connections])
    del stops

In [74]:
conns_with_hour.write_parquet(
    f"{path}/hexagon_network/{VARIANT}/{RES}/partials/connections_with_hour.parquet"
)

In [68]:
connections.filter(pl.col.device_id == 6513).tail(10)

device_id,day,hour,source,target,weight
i64,str,str,str,str,u32
6513,"""2025-06-18""","""06""","""871e037a1ffffff""","""871e03713ffffff""",1
6513,"""2025-06-18""","""08""","""871e03713ffffff""","""871e037a1ffffff""",1
6513,"""2025-06-18""","""10""","""871e037a1ffffff""","""871e03713ffffff""",1
6513,"""2025-06-18""","""12""","""871e03713ffffff""","""871e037a1ffffff""",1
6513,"""2025-06-18""","""14""","""871e037a1ffffff""","""871e03713ffffff""",1
6513,"""2025-06-18""","""15""","""871e03713ffffff""","""871e037a3ffffff""",1
6513,"""2025-06-18""","""17""","""871e037a3ffffff""","""871e037aaffffff""",1
6513,"""2025-06-18""","""19""","""871e037aaffffff""","""871e037a1ffffff""",1
6513,"""2025-06-18""","""20""","""871e037a1ffffff""","""871e037aaffffff""",1


In [65]:
connections_.filter(pl.col.device_id == 6513)

device_id,day,source,target,weight
i64,str,str,str,u32
6513,"""2025-06-16""","""871e03713ffffff""","""871e037a1ffffff""",1
6513,"""2025-06-17""","""871e03713ffffff""","""871e037a1ffffff""",5
6513,"""2025-06-17""","""871e037a1ffffff""","""871e03713ffffff""",5
6513,"""2025-06-18""","""871e03713ffffff""","""871e037a1ffffff""",2
6513,"""2025-06-18""","""871e03713ffffff""","""871e037a3ffffff""",1
6513,"""2025-06-18""","""871e037aaffffff""","""871e037a1ffffff""",2
6513,"""2025-06-18""","""871e037a1ffffff""","""871e03713ffffff""",3
6513,"""2025-06-18""","""871e037a3ffffff""","""871e037aaffffff""",1
6513,"""2025-06-18""","""871e037a1ffffff""","""871e037aaffffff""",1
